# Diffusion-Based Super-Resolution (SR3-Style)
### DIV2K Dataset · 4× Upscaling · 300 Epochs · DDIM Inference

In [ ]:
# ─────────────────────────────────────────────
# CELL 1: Setup and Imports
# ─────────────────────────────────────────────
import os
import glob
import math
import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt
from tensorflow import keras
from tensorflow.keras import layers

AUTOTUNE      = tf.data.AUTOTUNE
BATCH_SIZE    = 8
EPOCHS        = 300          # ← updated from 60
HR_CROP_SIZE  = 96
LR_CROP_SIZE  = 24
SCALE         = 4
TIMESTEPS     = 1000

print(f"TensorFlow version: {tf.__version__}")
print(f"Keras version:      {keras.__version__}")
print(f"GPUs:               {tf.config.list_physical_devices('GPU')}")

In [ ]:
# ─────────────────────────────────────────────
# CELL 2: Dataset Paths
# ─────────────────────────────────────────────
TRAIN_HR_DIR = "/kaggle/input/divrk-dataset/DIV2K_train_HR/DIV2K_train_HR"
TRAIN_LR_DIR = "/kaggle/input/divrk-dataset/DIV2K_train_LR_bicubic_X4/DIV2K_train_LR_bicubic/X4"
VALID_HR_DIR = "/kaggle/input/divrk-dataset/DIV2K_valid_HR/DIV2K_valid_HR"
VALID_LR_DIR = "/kaggle/input/divrk-dataset/DIV2K_valid_LR_bicubic_X4/DIV2K_valid_LR_bicubic/X4"

for path in [TRAIN_HR_DIR, TRAIN_LR_DIR, VALID_HR_DIR, VALID_LR_DIR]:
    exists = os.path.exists(path)
    count  = len(os.listdir(path)) if exists else 0
    print(f"{'✅' if exists else '❌'} {path}")
    print(f"   → {count} files found\n")

In [ ]:
# ─────────────────────────────────────────────
# CELL 3: Build File Pairs
# ─────────────────────────────────────────────
def get_paired_paths(hr_dir, lr_dir):
    """Match HR and LR image paths by their numeric ID."""
    hr_paths = sorted(glob.glob(os.path.join(hr_dir, "*.png")))
    lr_paths = sorted(glob.glob(os.path.join(lr_dir, "*.png")))

    hr_dict = {}
    for p in hr_paths:
        file_id = os.path.basename(p).split(".")[0]
        hr_dict[file_id] = p

    lr_dict = {}
    for p in lr_paths:
        file_id = os.path.basename(p).split("x")[0]
        lr_dict[file_id] = p

    paired_lr, paired_hr = [], []
    common_ids = sorted(set(hr_dict.keys()) & set(lr_dict.keys()))

    for file_id in common_ids:
        paired_lr.append(lr_dict[file_id])
        paired_hr.append(hr_dict[file_id])

    return paired_lr, paired_hr


train_lr_paths, train_hr_paths = get_paired_paths(TRAIN_HR_DIR, TRAIN_LR_DIR)
val_lr_paths,   val_hr_paths   = get_paired_paths(VALID_HR_DIR, VALID_LR_DIR)

print(f"Training pairs:   {len(train_lr_paths)}")
print(f"Validation pairs: {len(val_lr_paths)}")
print(f"\nSample pair:")
print(f"  LR: {os.path.basename(train_lr_paths[0])}")
print(f"  HR: {os.path.basename(train_hr_paths[0])}")

In [ ]:
# ─────────────────────────────────────────────
# CELL 4: Data Pipeline
# ─────────────────────────────────────────────
def load_image_pair(lr_path, hr_path):
    """Load and decode a LR-HR image pair from file paths."""
    lr_img = tf.io.read_file(lr_path)
    hr_img = tf.io.read_file(hr_path)
    lr_img = tf.image.decode_png(lr_img, channels=3)
    hr_img = tf.image.decode_png(hr_img, channels=3)
    return lr_img, hr_img


def random_crop(lr_img, hr_img, hr_crop_size=96, scale=4):
    """Randomly crop corresponding regions from LR and HR images."""
    lr_crop_size = hr_crop_size // scale
    lr_shape = tf.shape(lr_img)[:2]

    lr_h = tf.random.uniform((), 0, lr_shape[0] - lr_crop_size + 1, dtype=tf.int32)
    lr_w = tf.random.uniform((), 0, lr_shape[1] - lr_crop_size + 1, dtype=tf.int32)

    hr_h = lr_h * scale
    hr_w = lr_w * scale

    lr_cropped = lr_img[lr_h:lr_h + lr_crop_size, lr_w:lr_w + lr_crop_size]
    hr_cropped = hr_img[hr_h:hr_h + hr_crop_size, hr_w:hr_w + hr_crop_size]

    lr_cropped = tf.ensure_shape(lr_cropped, [lr_crop_size, lr_crop_size, 3])
    hr_cropped = tf.ensure_shape(hr_cropped, [hr_crop_size, hr_crop_size, 3])

    return lr_cropped, hr_cropped


def random_flip_and_rotate(lr_img, lr_up_img, hr_img):
    """Random horizontal flip and 90 degree rotation augmentation."""
    if tf.random.uniform(()) > 0.5:
        lr_img    = tf.image.flip_left_right(lr_img)
        lr_up_img = tf.image.flip_left_right(lr_up_img)
        hr_img    = tf.image.flip_left_right(hr_img)

    k = tf.random.uniform((), maxval=4, dtype=tf.int32)
    lr_img    = tf.image.rot90(lr_img,    k)
    lr_up_img = tf.image.rot90(lr_up_img, k)
    hr_img    = tf.image.rot90(hr_img,    k)

    return lr_img, lr_up_img, hr_img


def normalize_and_upsample(lr_img, hr_img):
    """Normalize to [0,1] and create bicubic-upsampled LR."""
    lr_img = tf.cast(lr_img, tf.float32) / 255.0
    hr_img = tf.cast(hr_img, tf.float32) / 255.0

    lr_upsampled = tf.image.resize(
        lr_img,
        [HR_CROP_SIZE, HR_CROP_SIZE],
        method="bicubic"
    )
    lr_upsampled = tf.clip_by_value(lr_upsampled, 0.0, 1.0)
    lr_upsampled = tf.ensure_shape(lr_upsampled, [HR_CROP_SIZE, HR_CROP_SIZE, 3])

    return lr_img, lr_upsampled, hr_img


def create_dataset(lr_paths, hr_paths, training=True):
    """Create tf.data.Dataset pipeline returning (lr, lr_upsampled, hr)."""
    ds = tf.data.Dataset.from_tensor_slices((lr_paths, hr_paths))

    if training:
        ds = ds.shuffle(buffer_size=len(lr_paths))

    ds = ds.map(load_image_pair, num_parallel_calls=AUTOTUNE)
    ds = ds.map(
        lambda lr, hr: random_crop(lr, hr, HR_CROP_SIZE, SCALE),
        num_parallel_calls=AUTOTUNE
    )
    ds = ds.map(normalize_and_upsample, num_parallel_calls=AUTOTUNE)

    if training:
        ds = ds.map(random_flip_and_rotate, num_parallel_calls=AUTOTUNE)

    ds = ds.batch(BATCH_SIZE)

    if training:
        ds = ds.repeat()

    ds = ds.prefetch(AUTOTUNE)
    return ds


train_ds = create_dataset(train_lr_paths, train_hr_paths, training=True)
val_ds   = create_dataset(val_lr_paths,   val_hr_paths,   training=False)

for lr_batch, lr_up_batch, hr_batch in train_ds.take(1):
    print(f"LR batch shape:           {lr_batch.shape}")
    print(f"LR upsampled batch shape: {lr_up_batch.shape}")
    print(f"HR batch shape:           {hr_batch.shape}")
    print(f"LR range:    [{lr_batch.numpy().min():.3f}, {lr_batch.numpy().max():.3f}]")
    print(f"LR_up range: [{lr_up_batch.numpy().min():.3f}, {lr_up_batch.numpy().max():.3f}]")
    print(f"HR range:    [{hr_batch.numpy().min():.3f}, {hr_batch.numpy().max():.3f}]")

print("\n✅ Datasets created successfully!")

In [ ]:
# ─────────────────────────────────────────────
# CELL 5: Visualize Dataset
# ─────────────────────────────────────────────
lr_batch, lr_up_batch, hr_batch = next(iter(train_ds))

fig, axes = plt.subplots(3, 4, figsize=(16, 12))
fig.suptitle("Training Data: LR → Bicubic Upsample → HR Ground Truth", fontsize=16)

for i in range(4):
    axes[0, i].imshow(lr_batch[i].numpy())
    axes[0, i].set_title(f"LR {lr_batch[i].shape}")
    axes[0, i].axis("off")

    axes[1, i].imshow(lr_up_batch[i].numpy())
    axes[1, i].set_title(f"LR Upsampled {lr_up_batch[i].shape}")
    axes[1, i].axis("off")

    axes[2, i].imshow(hr_batch[i].numpy())
    axes[2, i].set_title(f"HR {hr_batch[i].shape}")
    axes[2, i].axis("off")

plt.tight_layout()
plt.show()

In [ ]:
# ─────────────────────────────────────────────
# CELL 6: Diffusion Schedule
# ─────────────────────────────────────────────
def cosine_beta_schedule(timesteps, s=0.008):
    """Cosine noise schedule (Nichol & Dhariwal 2021)."""
    steps = np.arange(timesteps + 1, dtype=np.float64)
    f = np.cos((steps / timesteps + s) / (1 + s) * np.pi * 0.5) ** 2
    alphas_cumprod = f / f[0]
    betas = 1 - alphas_cumprod[1:] / alphas_cumprod[:-1]
    return np.clip(betas, 0.0001, 0.9999)


betas               = cosine_beta_schedule(TIMESTEPS)
alphas              = 1.0 - betas
alphas_cumprod      = np.cumprod(alphas)
alphas_cumprod_prev = np.concatenate([[1.0], alphas_cumprod[:-1]])

sqrt_alphas_cumprod           = np.sqrt(alphas_cumprod).astype(np.float32)
sqrt_one_minus_alphas_cumprod = np.sqrt(1.0 - alphas_cumprod).astype(np.float32)
sqrt_recip_alphas             = (1.0 / np.sqrt(alphas)).astype(np.float32)
posterior_variance            = (betas * (1.0 - alphas_cumprod_prev) / (1.0 - alphas_cumprod)).astype(np.float32)

sqrt_alphas_cumprod_tf           = tf.constant(sqrt_alphas_cumprod)
sqrt_one_minus_alphas_cumprod_tf = tf.constant(sqrt_one_minus_alphas_cumprod)
posterior_variance_tf            = tf.constant(posterior_variance)
betas_tf                         = tf.constant(betas.astype(np.float32))
sqrt_recip_alphas_tf             = tf.constant(sqrt_recip_alphas)
alphas_cumprod_tf                = tf.constant(alphas_cumprod.astype(np.float32))

print(f"β range: [{betas[0]:.6f}, {betas[-1]:.6f}]")
print(f"ᾱ range: [{alphas_cumprod[-1]:.6f}, {alphas_cumprod[0]:.6f}]")

fig, axes = plt.subplots(1, 3, figsize=(18, 4))
axes[0].plot(betas)
axes[0].set_title("β_t (Noise added per step)")
axes[0].set_xlabel("Timestep")

axes[1].plot(alphas_cumprod)
axes[1].set_title("ᾱ_t (Cumulative signal retained)")
axes[1].set_xlabel("Timestep")

axes[2].plot(sqrt_one_minus_alphas_cumprod)
axes[2].set_title("√(1-ᾱ_t) (Noise level)")
axes[2].set_xlabel("Timestep")

plt.tight_layout()
plt.show()

In [ ]:
# ─────────────────────────────────────────────
# CELL 7: Forward Diffusion
# ─────────────────────────────────────────────
def extract(tensor, t, shape):
    """Gather values from tensor at indices t, reshape for broadcasting."""
    batch_size = tf.shape(t)[0]
    values = tf.gather(tensor, t)
    return tf.reshape(values, [batch_size] + [1] * (len(shape) - 1))


def forward_diffusion(x_0, t, noise=None):
    """
    Add noise to clean image x_0 at timestep t.
    q(x_t | x_0) = sqrt(alpha_bar_t) * x_0 + sqrt(1 - alpha_bar_t) * noise
    """
    if noise is None:
        noise = tf.random.normal(tf.shape(x_0))

    sqrt_alpha       = extract(sqrt_alphas_cumprod_tf,           t, x_0.shape)
    sqrt_one_minus   = extract(sqrt_one_minus_alphas_cumprod_tf, t, x_0.shape)

    x_t = sqrt_alpha * x_0 + sqrt_one_minus * noise
    return x_t, noise


# Visualize forward diffusion on a sample
lr_sample, lr_up_sample, hr_sample = next(iter(val_ds))
sample_img = hr_sample[0:1]

fig, axes = plt.subplots(1, 6, figsize=(24, 4))
fig.suptitle("Forward Diffusion Process: Gradually Adding Noise to HR Image", fontsize=14)

timesteps_to_show = [0, 50, 200, 500, 800, 999]
for i, t_val in enumerate(timesteps_to_show):
    t = tf.constant([t_val])
    noisy, _ = forward_diffusion(sample_img, t)
    noisy_clipped = tf.clip_by_value(noisy[0], 0, 1)
    axes[i].imshow(noisy_clipped.numpy())
    axes[i].set_title(f"t = {t_val}")
    axes[i].axis("off")

plt.tight_layout()
plt.show()

In [ ]:
# ─────────────────────────────────────────────
# CELL 8: U-Net Building Blocks
# ─────────────────────────────────────────────
def sinusoidal_embedding(timesteps, dim=128):
    """Sinusoidal positional embedding for timesteps."""
    half_dim = dim // 2
    emb = math.log(10000.0) / (half_dim - 1)
    emb = tf.exp(tf.range(half_dim, dtype=tf.float32) * -emb)
    emb = tf.cast(timesteps, tf.float32)[:, None] * emb[None, :]
    return tf.concat([tf.sin(emb), tf.cos(emb)], axis=-1)


class TimeEmbeddingLayer(layers.Layer):
    """MLP to project time embeddings to higher dimension."""

    def __init__(self, dim, **kwargs):
        super().__init__(**kwargs)
        self.dim    = dim
        self.dense1 = layers.Dense(dim * 4, activation="swish")
        self.dense2 = layers.Dense(dim * 4)

    def call(self, t):
        emb = sinusoidal_embedding(t, self.dim)
        return self.dense2(self.dense1(emb))


class ConditionalResBlock(layers.Layer):
    """Residual block with timestep conditioning."""

    def __init__(self, filters, groups=8, **kwargs):
        super().__init__(**kwargs)
        self.filters       = filters
        self.norm1         = layers.GroupNormalization(groups=min(groups, filters))
        self.act1          = layers.Activation("swish")
        self.conv1         = layers.Conv2D(filters, 3, padding="same")
        self.time_proj     = layers.Dense(filters)
        self.time_act      = layers.Activation("swish")
        self.norm2         = layers.GroupNormalization(groups=min(groups, filters))
        self.act2          = layers.Activation("swish")
        self.conv2         = layers.Conv2D(filters, 3, padding="same")
        self.match_channels = None

    def build(self, input_shape):
        if input_shape[-1] != self.filters:
            self.match_channels = layers.Conv2D(self.filters, 1)
        super().build(input_shape)

    def call(self, x, time_emb):
        residual = x

        h = self.norm1(x)
        h = self.act1(h)
        h = self.conv1(h)

        t = self.time_act(time_emb)
        t = self.time_proj(t)
        h = h + t[:, None, None, :]

        h = self.norm2(h)
        h = self.act2(h)
        h = self.conv2(h)

        if self.match_channels is not None:
            residual = self.match_channels(residual)

        return h + residual


class SelfAttentionBlock(layers.Layer):
    """Self-attention for capturing long-range spatial dependencies."""

    def __init__(self, filters, groups=8, **kwargs):
        super().__init__(**kwargs)
        self.filters = filters
        self.norm    = layers.GroupNormalization(groups=min(groups, filters))
        self.attn    = layers.MultiHeadAttention(
            num_heads=4, key_dim=max(filters // 4, 1)
        )

    def call(self, x):
        residual = x
        x_norm   = self.norm(x)

        shape = tf.shape(x_norm)
        b, h, w, c = shape[0], shape[1], shape[2], shape[3]

        x_flat = tf.reshape(x_norm, [b, h * w, c])
        x_attn = self.attn(x_flat, x_flat)
        x_out  = tf.reshape(x_attn, [b, h, w, c])

        return x_out + residual


print("✅ Building blocks defined (Keras 3 compatible)")

In [ ]:
# ─────────────────────────────────────────────
# CELL 9: Build the U-Net
# ─────────────────────────────────────────────
def build_unet(
    image_size=HR_CROP_SIZE,
    in_channels=6,
    out_channels=3,
    base_filters=64,
    channel_mults=(1, 2, 4),
    num_res_blocks=2,
    time_emb_dim=128,
    use_attention_at=(24,),
):
    """Build conditional U-Net for diffusion SR (Keras 3 compatible)."""

    # ── INPUTS ──
    image_input = layers.Input(
        shape=(image_size, image_size, in_channels),
        name="image_input"
    )
    time_input = layers.Input(shape=(), dtype=tf.int32, name="timestep")

    # ── TIME EMBEDDING ──
    time_emb = TimeEmbeddingLayer(time_emb_dim)(time_input)

    # ── INITIAL CONV ──
    h = layers.Conv2D(base_filters, 3, padding="same")(image_input)
    skips = [h]

    # ── ENCODER ──
    current_res = image_size

    for level, mult in enumerate(channel_mults):
        filters = base_filters * mult

        for _ in range(num_res_blocks):
            h = ConditionalResBlock(filters)(h, time_emb)
            if current_res in use_attention_at:
                h = SelfAttentionBlock(filters)(h)
            skips.append(h)

        if level < len(channel_mults) - 1:
            h = layers.Conv2D(filters, 3, strides=2, padding="same")(h)
            current_res //= 2
            skips.append(h)

    # ── BOTTLENECK ──
    h = ConditionalResBlock(filters)(h, time_emb)
    h = SelfAttentionBlock(filters)(h)
    h = ConditionalResBlock(filters)(h, time_emb)

    # ── DECODER ──
    for level in reversed(range(len(channel_mults))):
        filters = base_filters * channel_mults[level]

        for _ in range(num_res_blocks + 1):
            skip = skips.pop()
            h = layers.Concatenate()([h, skip])
            h = ConditionalResBlock(filters)(h, time_emb)
            if current_res in use_attention_at:
                h = SelfAttentionBlock(filters)(h)

        if level > 0:
            h = layers.Conv2DTranspose(filters, 3, strides=2, padding="same")(h)
            current_res *= 2

    assert len(skips) == 0, f"Unused skip connections: {len(skips)}"

    # ── OUTPUT ──
    h = layers.GroupNormalization(groups=8)(h)
    h = layers.Activation("swish")(h)
    output = layers.Conv2D(
        out_channels, 3, padding="same",
        kernel_initializer="zeros"
    )(h)

    model = keras.Model([image_input, time_input], output, name="Conditional_UNet")
    return model


unet = build_unet(
    image_size=HR_CROP_SIZE,
    base_filters=64,
    channel_mults=(1, 2, 4),
    num_res_blocks=2,
    time_emb_dim=128,
    use_attention_at=(24,),
)

unet.summary()
print(f"\n✅ Total parameters: {unet.count_params():,}")

In [ ]:
# ─────────────────────────────────────────────
# CELL 10: DiffusionSR Training Model
# ─────────────────────────────────────────────
class DiffusionSR(keras.Model):
    """Diffusion-based Super-Resolution model wrapper."""

    def __init__(self, unet, **kwargs):
        super().__init__(**kwargs)
        self.unet         = unet
        self.loss_tracker = keras.metrics.Mean(name="loss")

    @property
    def metrics(self):
        return [self.loss_tracker]

    def build(self, input_shape):
        self.built = True

    def _shared_step(self, data, training):
        lr_images, lr_upsampled, hr_images = data
        batch_size = tf.shape(hr_images)[0]

        t     = tf.random.uniform((batch_size,), 0, TIMESTEPS, dtype=tf.int32)
        noise = tf.random.normal(tf.shape(hr_images))
        x_t, _ = forward_diffusion(hr_images, t, noise)

        model_input = tf.concat([x_t, lr_upsampled], axis=-1)

        if training:
            with tf.GradientTape() as tape:
                predicted_noise = self.unet([model_input, t], training=True)
                loss = tf.reduce_mean(tf.square(noise - predicted_noise))

            gradients = tape.gradient(loss, self.unet.trainable_variables)
            self.optimizer.apply_gradients(
                zip(gradients, self.unet.trainable_variables)
            )
        else:
            predicted_noise = self.unet([model_input, t], training=False)
            loss = tf.reduce_mean(tf.square(noise - predicted_noise))

        return loss

    def train_step(self, data):
        loss = self._shared_step(data, training=True)
        self.loss_tracker.update_state(loss)
        return {"loss": self.loss_tracker.result()}

    def test_step(self, data):
        loss = self._shared_step(data, training=False)
        self.loss_tracker.update_state(loss)
        return {"loss": self.loss_tracker.result()}


diffusion_model = DiffusionSR(unet)

diffusion_model.compile(
    optimizer=keras.optimizers.Adam(
        learning_rate=2e-4,
        clipnorm=1.0
    )
)

# Build the model by running one batch
for sample_data in train_ds.take(1):
    diffusion_model.train_step(sample_data)

print("✅ Diffusion SR model compiled and built!")

In [ ]:
# ─────────────────────────────────────────────
# CELL 11: Callbacks
# ─────────────────────────────────────────────
class SaveUNetWeights(keras.callbacks.Callback):
    """Save the inner UNet weights directly."""

    def __init__(self, filepath, monitor="val_loss", save_best_only=True):
        super().__init__()
        self.filepath       = filepath
        self.monitor        = monitor
        self.save_best_only = save_best_only
        self.best           = float("inf")

    def on_epoch_end(self, epoch, logs=None):
        current_val = logs.get(self.monitor)
        if current_val is None:
            return

        if self.save_best_only:
            if current_val < self.best:
                self.best = current_val
                self.model.unet.save_weights(self.filepath)
                print(f"\n✅ Epoch {epoch+1}: {self.monitor} improved to "
                      f"{current_val:.5f}, saving UNet weights to {self.filepath}")
        else:
            self.model.unet.save_weights(self.filepath)


best_weights_path = "diffusion_sr_best.weights.h5"

save_best_cb = SaveUNetWeights(
    filepath=best_weights_path,
    monitor="val_loss",
    save_best_only=True,
)

lr_reduce_cb = keras.callbacks.ReduceLROnPlateau(
    monitor="val_loss",
    factor=0.5,
    patience=10,        # ← increased from 8 (scaled for 300-epoch run)
    min_lr=1e-6,
    verbose=1,
)

early_stop_cb = keras.callbacks.EarlyStopping(
    monitor="val_loss",
    patience=40,        # ← increased from 20 (allows longer plateau exploration)
    restore_best_weights=False,
    verbose=1,
)

callbacks = [save_best_cb, lr_reduce_cb, early_stop_cb]
print("✅ Callbacks defined!")

In [ ]:
# ─────────────────────────────────────────────
# CELL 12: Train
# ─────────────────────────────────────────────
history = diffusion_model.fit(
    train_ds,
    epochs=EPOCHS,            # 300
    steps_per_epoch=400,      # ← increased from 200 → 120,000 total gradient updates
    validation_data=val_ds,
    callbacks=callbacks,
)

In [ ]:
# ─────────────────────────────────────────────
# CELL 13: Training Curves
# ─────────────────────────────────────────────
plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
plt.plot(history.history["loss"],     label="Train Loss", color="blue")
plt.plot(history.history["val_loss"], label="Val Loss",   color="orange")
plt.xlabel("Epoch")
plt.ylabel("MSE (Noise Prediction)")
plt.title("Training & Validation Loss")
plt.legend()
plt.grid(True, alpha=0.3)

plt.subplot(1, 2, 2)
plt.plot(history.history["loss"],     label="Train Loss", color="blue")
plt.plot(history.history["val_loss"], label="Val Loss",   color="orange")
plt.xlabel("Epoch")
plt.ylabel("MSE (Noise Prediction)")
plt.title("Loss (Log Scale)")
plt.yscale("log")
plt.legend()
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# ─────────────────────────────────────────────
# CELL 14: Load Best Weights
# ─────────────────────────────────────────────
unet.load_weights(best_weights_path)
print("✅ Best UNet weights loaded!")

In [ ]:
# ─────────────────────────────────────────────
# CELL 15: Plotting Helper Functions
# ─────────────────────────────────────────────
def plot_sr_results(lr_img, sr_img, hr_img=None, bicubic_img=None):
    """Display comparison of LR, Bicubic, Diffusion SR, and Ground Truth."""

    images = [("Low Resolution", lr_img)]

    if bicubic_img is not None:
        images.append(("Bicubic Upscale", bicubic_img))

    images.append(("Diffusion SR", sr_img))

    if hr_img is not None:
        images.append(("Ground Truth HR", hr_img))

    num_cols = len(images)
    fig, axes = plt.subplots(1, num_cols, figsize=(7 * num_cols, 7))

    if num_cols == 1:
        axes = [axes]

    for i, (title, img) in enumerate(images):
        if hasattr(img, 'numpy'):
            img = img.numpy()

        if img.dtype != np.uint8:
            if img.max() <= 1.0:
                img = (img * 255).clip(0, 255).astype(np.uint8)
            else:
                img = img.clip(0, 255).astype(np.uint8)

        axes[i].imshow(img)
        axes[i].set_title(f"{title}\n{img.shape}", fontsize=12)
        axes[i].axis("off")

    plt.tight_layout()
    plt.show()


print("✅ Plotting functions defined!")

In [ ]:
# ─────────────────────────────────────────────
# CELL 16: DDIM Sampling
# ─────────────────────────────────────────────
def safe_resize_bicubic(image, target_size):
    """Bicubic resize that works on CPU."""
    try:
        result = tf.image.resize(image, target_size, method="bicubic")
    except Exception:
        result = tf.image.resize(image, target_size, method="bilinear")
    return tf.clip_by_value(result, 0.0, 1.0)


def ddim_sample(unet, lr_image, num_steps=50, eta=0.0, verbose=True):
    """
    DDIM sampling for a single 24x24 LR patch.
    Output: 96x96 SR image (uint8).
    """
    lr_shape = tf.shape(lr_image)
    hr_h, hr_w = lr_shape[0] * SCALE, lr_shape[1] * SCALE

    lr_up = safe_resize_bicubic(lr_image[None], [hr_h, hr_w])

    timesteps = np.linspace(TIMESTEPS - 1, 0, num_steps, dtype=np.int64)
    x_t = tf.random.normal((1, hr_h, hr_w, 3))

    for i in range(len(timesteps)):
        t       = int(timesteps[i])
        t_batch = tf.fill([1], t)

        if verbose and i % 10 == 0:
            print(f"  DDIM step {i+1}/{num_steps} (t={t})...")

        model_input = tf.concat([x_t, lr_up], axis=-1)
        pred_noise  = unet([model_input, t_batch], training=False)

        alpha_t    = float(alphas_cumprod[t])
        alpha_prev = float(alphas_cumprod[int(timesteps[i+1])]) if i + 1 < len(timesteps) else 1.0

        pred_x0 = (x_t - math.sqrt(1 - alpha_t) * pred_noise) / math.sqrt(alpha_t)
        pred_x0 = tf.clip_by_value(pred_x0, -1.0, 2.0)

        if (1 - alpha_t) > 0 and alpha_prev < alpha_t:
            sigma = eta * math.sqrt((1 - alpha_prev) / (1 - alpha_t)) * math.sqrt(1 - alpha_t / alpha_prev)
        else:
            sigma = 0.0

        dir_xt = math.sqrt(max(1 - alpha_prev - sigma**2, 0)) * pred_noise
        noise  = tf.random.normal(tf.shape(x_t)) if i < len(timesteps) - 1 and eta > 0 else 0.0
        x_t    = math.sqrt(alpha_prev) * pred_x0 + dir_xt + sigma * noise

    sr = tf.clip_by_value(x_t[0], 0.0, 1.0)
    sr = tf.cast(sr * 255, tf.uint8)
    return sr


print("✅ DDIM sampling function defined!")

In [ ]:
# ─────────────────────────────────────────────
# CELL 17: Tiled Inference for Any Image Size
# ─────────────────────────────────────────────
def ddim_sample_tiled(unet, lr_image, tile_size=24, num_steps=50, eta=0.0, verbose=True):
    """
    DDIM sampling with tiled processing for ANY input size.
    Splits LR image into tile_size x tile_size patches,
    upscales each 4x, then stitches back together.
    """
    hr_tile_size = tile_size * SCALE

    lr_h, lr_w = lr_image.shape[0], lr_image.shape[1]
    hr_h, hr_w = lr_h * SCALE, lr_w * SCALE

    pad_h = (tile_size - lr_h % tile_size) % tile_size
    pad_w = (tile_size - lr_w % tile_size) % tile_size
    lr_padded = tf.pad(lr_image, [[0, pad_h], [0, pad_w], [0, 0]], mode="REFLECT")

    padded_h, padded_w = lr_padded.shape[0], lr_padded.shape[1]
    tiles_y = padded_h // tile_size
    tiles_x = padded_w // tile_size

    if verbose:
        total_tiles = tiles_y * tiles_x
        print(f"  Image: {lr_h}x{lr_w} → padded: {padded_h}x{padded_w}")
        print(f"  Tiles: {tiles_y}x{tiles_x} = {total_tiles} tiles")

    sr_padded = np.zeros((padded_h * SCALE, padded_w * SCALE, 3), dtype=np.float32)

    tile_count = 0
    for ty in range(tiles_y):
        for tx in range(tiles_x):
            tile_count += 1
            if verbose:
                print(f"  Processing tile {tile_count}/{tiles_y * tiles_x}...")

            y_start = ty * tile_size
            x_start = tx * tile_size
            lr_tile = lr_padded[y_start:y_start + tile_size,
                                x_start:x_start + tile_size]

            lr_up = safe_resize_bicubic(lr_tile[None], [hr_tile_size, hr_tile_size])
            x_t   = tf.random.normal((1, hr_tile_size, hr_tile_size, 3))

            timesteps_arr = np.linspace(TIMESTEPS - 1, 0, num_steps, dtype=np.int64)

            for i in range(len(timesteps_arr)):
                t       = int(timesteps_arr[i])
                t_batch = tf.fill([1], t)

                model_input = tf.concat([x_t, lr_up], axis=-1)
                pred_noise  = unet([model_input, t_batch], training=False)

                alpha_t    = float(alphas_cumprod[t])
                alpha_prev = float(alphas_cumprod[int(timesteps_arr[i+1])]) if i + 1 < len(timesteps_arr) else 1.0

                pred_x0 = (x_t - math.sqrt(1 - alpha_t) * pred_noise) / math.sqrt(alpha_t)
                pred_x0 = tf.clip_by_value(pred_x0, -1.0, 2.0)

                if (1 - alpha_t) > 0 and alpha_prev < alpha_t:
                    sigma = eta * math.sqrt((1 - alpha_prev) / (1 - alpha_t)) * math.sqrt(1 - alpha_t / alpha_prev)
                else:
                    sigma = 0.0

                dir_xt = math.sqrt(max(1 - alpha_prev - sigma**2, 0)) * pred_noise
                noise  = tf.random.normal(tf.shape(x_t)) if i < len(timesteps_arr) - 1 and eta > 0 else 0.0
                x_t    = math.sqrt(alpha_prev) * pred_x0 + dir_xt + sigma * noise

            sr_tile = tf.clip_by_value(x_t[0], 0.0, 1.0).numpy()
            hr_y = ty * hr_tile_size
            hr_x = tx * hr_tile_size
            sr_padded[hr_y:hr_y + hr_tile_size, hr_x:hr_x + hr_tile_size] = sr_tile

    sr_final = sr_padded[:hr_h, :hr_w]
    sr_final = (sr_final * 255).clip(0, 255).astype(np.uint8)
    return sr_final


print("✅ Tiled inference function defined!")

In [ ]:
# ─────────────────────────────────────────────
# CELL 18: Upscale Validation Images (Small Crops)
# ─────────────────────────────────────────────
print("🔄 Generating Super-Resolution images (small crops)...\n")

raw_val_ds = tf.data.Dataset.from_tensor_slices((val_lr_paths, val_hr_paths))
raw_val_ds = raw_val_ds.map(load_image_pair, num_parallel_calls=AUTOTUNE)

for idx, (lr_raw, hr_raw) in enumerate(raw_val_ds.take(6)):
    print(f"━━━ Image {idx + 1} ━━━")

    # 24x24 LR crop → 96x96 SR (matches model input size directly)
    lr_crop = tf.image.random_crop(lr_raw, (LR_CROP_SIZE, LR_CROP_SIZE, 3))
    lr_norm = tf.cast(lr_crop, tf.float32) / 255.0

    # Bicubic baseline
    bicubic = tf.image.resize(lr_crop, [HR_CROP_SIZE, HR_CROP_SIZE], method="bicubic")
    bicubic = tf.clip_by_value(bicubic, 0, 255)
    bicubic = tf.cast(bicubic, tf.uint8)

    # Diffusion SR
    sr_image = ddim_sample(unet, lr_norm, num_steps=50, verbose=True)

    print(f"  LR: {lr_crop.shape} → SR: {sr_image.shape}")

    plot_sr_results(
        lr_img=lr_crop.numpy(),
        sr_img=sr_image.numpy(),
        bicubic_img=bicubic.numpy()
    )
    print()

In [ ]:
# ─────────────────────────────────────────────
# CELL 19: Upscale Larger Validation Images (Tiled)
# ─────────────────────────────────────────────
print("🔄 Generating Super-Resolution images (tiled, larger crops)...\n")

raw_val_ds2 = tf.data.Dataset.from_tensor_slices((val_lr_paths, val_hr_paths))
raw_val_ds2 = raw_val_ds2.map(load_image_pair, num_parallel_calls=AUTOTUNE)

for idx, (lr_raw, hr_raw) in enumerate(raw_val_ds2.take(4)):
    print(f"━━━ Image {idx + 1} ━━━")

    # 72x72 LR crop → 288x288 SR (processed as 3x3 = 9 tiles)
    crop_size = 72  # Must be divisible by 24
    lr_crop   = tf.image.random_crop(lr_raw, (crop_size, crop_size, 3))
    lr_norm   = tf.cast(lr_crop, tf.float32) / 255.0

    hr_size = crop_size * SCALE
    bicubic = tf.image.resize(lr_crop, [hr_size, hr_size], method="bicubic")
    bicubic = tf.clip_by_value(bicubic, 0, 255)
    bicubic = tf.cast(bicubic, tf.uint8)

    sr_image = ddim_sample_tiled(
        unet, lr_norm,
        tile_size=LR_CROP_SIZE,
        num_steps=50,
        verbose=True
    )

    print(f"  LR: {lr_crop.shape} → SR: {sr_image.shape}")

    plot_sr_results(
        lr_img=lr_crop.numpy(),
        sr_img=sr_image,
        bicubic_img=bicubic.numpy()
    )
    print()

In [ ]:
# ─────────────────────────────────────────────
# CELL 20: PSNR Evaluation
# ─────────────────────────────────────────────
def evaluate_psnr(unet, val_lr_paths, val_hr_paths, num_images=10, num_ddim_steps=50):
    """Evaluate PSNR on validation images."""

    psnr_diffusion = []
    psnr_bicubic   = []

    ds = tf.data.Dataset.from_tensor_slices(
        (val_lr_paths[:num_images], val_hr_paths[:num_images])
    )
    ds = ds.map(load_image_pair)

    for i, (lr_img, hr_img) in enumerate(ds):
        print(f"Evaluating image {i+1}/{num_images}...")

        lr_shape = tf.shape(lr_img)[:2]
        start_h  = (lr_shape[0] - LR_CROP_SIZE) // 2
        start_w  = (lr_shape[1] - LR_CROP_SIZE) // 2

        lr_crop = lr_img[start_h:start_h + LR_CROP_SIZE,
                         start_w:start_w + LR_CROP_SIZE]
        hr_crop = hr_img[start_h * SCALE:start_h * SCALE + HR_CROP_SIZE,
                         start_w * SCALE:start_w * SCALE + HR_CROP_SIZE]

        lr_norm = tf.cast(lr_crop, tf.float32) / 255.0

        # Diffusion SR
        sr = ddim_sample(unet, lr_norm, num_steps=num_ddim_steps, verbose=False)

        # Bicubic baseline
        bicubic = tf.image.resize(lr_crop[None], [HR_CROP_SIZE, HR_CROP_SIZE], method="bicubic")
        bicubic = tf.clip_by_value(bicubic, 0, 255)
        bicubic = tf.cast(bicubic, tf.uint8)

        # PSNR
        hr_f  = tf.cast(hr_crop[None], tf.float32)
        sr_f  = tf.cast(sr[None],      tf.float32)
        bic_f = tf.cast(bicubic,       tf.float32)

        p_diff = tf.image.psnr(hr_f, sr_f,  max_val=255.0)[0].numpy()
        p_bic  = tf.image.psnr(hr_f, bic_f, max_val=255.0)[0].numpy()

        psnr_diffusion.append(p_diff)
        psnr_bicubic.append(p_bic)

        print(f"  Diffusion: {p_diff:.2f} dB | Bicubic: {p_bic:.2f} dB")

    print(f"\n{'='*50}")
    print(f"Average Diffusion PSNR: {np.mean(psnr_diffusion):.2f} dB")
    print(f"Average Bicubic PSNR:   {np.mean(psnr_bicubic):.2f} dB")
    print(f"Improvement:            +{np.mean(psnr_diffusion) - np.mean(psnr_bicubic):.2f} dB")

    return psnr_diffusion, psnr_bicubic


psnr_diff, psnr_bic = evaluate_psnr(unet, val_lr_paths, val_hr_paths, num_images=10)

In [ ]:
# ─────────────────────────────────────────────
# CELL 21: PSNR Bar Chart
# ─────────────────────────────────────────────
plt.figure(figsize=(12, 5))

x     = np.arange(len(psnr_diff))
width = 0.35

plt.bar(x - width/2, psnr_bic,  width, label='Bicubic',      color='orange', alpha=0.8)
plt.bar(x + width/2, psnr_diff, width, label='Diffusion SR', color='blue',   alpha=0.8)

plt.xlabel('Image Index')
plt.ylabel('PSNR (dB)')
plt.title('PSNR Comparison: Bicubic vs Diffusion SR')
plt.xticks(x, [f"Img {i+1}" for i in x])
plt.legend()
plt.grid(True, alpha=0.3, axis='y')

plt.axhline(y=np.mean(psnr_bic),  color='orange', linestyle='--', alpha=0.5,
            label=f'Avg Bicubic: {np.mean(psnr_bic):.1f}')
plt.axhline(y=np.mean(psnr_diff), color='blue',   linestyle='--', alpha=0.5,
            label=f'Avg Diffusion: {np.mean(psnr_diff):.1f}')

plt.tight_layout()
plt.show()

In [ ]:
# ─────────────────────────────────────────────
# CELL 22: Visualize Denoising Process
# ─────────────────────────────────────────────
def visualize_denoising(unet, lr_image, num_steps=50, show_every=10):
    """Show how diffusion iteratively denoises from noise to SR image."""

    hr_h, hr_w = LR_CROP_SIZE * SCALE, LR_CROP_SIZE * SCALE

    lr_up = safe_resize_bicubic(lr_image[None], [hr_h, hr_w])

    timesteps_arr = np.linspace(TIMESTEPS - 1, 0, num_steps, dtype=np.int64)
    x_t = tf.random.normal((1, hr_h, hr_w, 3))

    snapshots = [("Noise\nt=T", tf.clip_by_value(x_t[0], 0, 1).numpy())]

    for i in range(len(timesteps_arr)):
        t       = int(timesteps_arr[i])
        t_batch = tf.fill([1], t)

        model_input = tf.concat([x_t, lr_up], axis=-1)
        pred_noise  = unet([model_input, t_batch], training=False)

        alpha_t    = float(alphas_cumprod[t])
        alpha_prev = float(alphas_cumprod[int(timesteps_arr[i+1])]) if i + 1 < len(timesteps_arr) else 1.0

        pred_x0 = (x_t - math.sqrt(1 - alpha_t) * pred_noise) / math.sqrt(alpha_t)
        pred_x0 = tf.clip_by_value(pred_x0, -1.0, 2.0)

        dir_xt = math.sqrt(max(1 - alpha_prev, 0)) * pred_noise
        x_t    = math.sqrt(alpha_prev) * pred_x0 + dir_xt

        if (i + 1) % show_every == 0 or i == len(timesteps_arr) - 1:
            label = f"Step {i+1}\nt={t}"
            snapshots.append((label, tf.clip_by_value(x_t[0], 0, 1).numpy()))

    # Plot
    n = len(snapshots) + 1
    fig, axes = plt.subplots(1, n, figsize=(3.5 * n, 3.5))
    fig.suptitle("Reverse Diffusion: Noise → Super-Resolution", fontsize=14)

    # Show LR input
    lr_display = lr_image.numpy()
    if lr_display.max() <= 1.0:
        lr_display = (lr_display * 255).clip(0, 255).astype(np.uint8)
    axes[0].imshow(lr_display)
    axes[0].set_title("LR Input")
    axes[0].axis("off")

    for idx, (label, img) in enumerate(snapshots):
        img_display = (img * 255).clip(0, 255).astype(np.uint8)
        axes[idx + 1].imshow(img_display)
        axes[idx + 1].set_title(label, fontsize=9)
        axes[idx + 1].axis("off")

    plt.tight_layout()
    plt.show()


# Run on a validation image
for lr_raw, hr_raw in raw_val_ds.take(1):
    lr_crop = tf.image.random_crop(lr_raw, (LR_CROP_SIZE, LR_CROP_SIZE, 3))
    lr_norm = tf.cast(lr_crop, tf.float32) / 255.0
    visualize_denoising(unet, lr_norm, num_steps=50, show_every=10)

In [ ]:
# ─────────────────────────────────────────────
# CELL 23: Summary
# ─────────────────────────────────────────────
print("=" * 60)
print("  DIFFUSION SUPER-RESOLUTION - PROJECT SUMMARY")
print("=" * 60)
print(f"""
Model:              Conditional U-Net (SR3-style)
Dataset:            DIV2K (800 train, 100 val)
Scale Factor:       {SCALE}x
Training Patches:   {LR_CROP_SIZE}x{LR_CROP_SIZE} LR → {HR_CROP_SIZE}x{HR_CROP_SIZE} HR
Diffusion Steps:    {TIMESTEPS} (cosine schedule)
Inference Steps:    50 (DDIM)
Parameters:         {unet.count_params():,}
Loss:               MSE (noise prediction)
Epochs:             {EPOCHS}
Batch Size:         {BATCH_SIZE}
Steps per Epoch:    400
Total Updates:      {EPOCHS * 400:,}

Results:
  Avg Bicubic PSNR:   {np.mean(psnr_bic):.2f} dB
  Avg Diffusion PSNR: {np.mean(psnr_diff):.2f} dB
  Improvement:        +{np.mean(psnr_diff) - np.mean(psnr_bic):.2f} dB
""")
print("✅ All done! 🎉")